# Toy generators

This notebooks contains a number of toy-classification generators

In [ ]:
import copy
import logging

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sts

from numpy.typing import NDArray

import ppu
from ppu.generator import Circular, Moons, RingBlobs
from ppu.methods.mlp import MLP
from ppu.methods.tracin import get_random_resampled_tracin, get_random_tracin, get_loss_over_grid, get_proba_over_grid
from ppu.viz import plot_dense_binary_scatter, plot_dense_scatter, plot_ellipse_from_cov, plot_pdf_contours, GridPlot

## Config

In [ ]:
n_samples = 10_000
colors = ppu.viz.set_plot_style(True)
rng = np.random.Generator(np.random.PCG64DXSM(42))

## Definitions

In [ ]:
gen = Moons(class_sep=0.5)
# gen = Circular()
# gen = RingBlobs()

X_train, y_train = gen.rvs(10000)

# ax = plot_dense_binary_scatter(X_train, y_train)

### Single point

In [ ]:
import torch

In [ ]:
mlp = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=40, frequency=3)
mlp.fit(X_train, y_train)

In [ ]:
grid = GridPlot(X=X_train, y=y_train, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=mlp)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=mlp)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
from ppu.methods.point_tracin import TracIn

In [ ]:
mlp._opt_lr = 1e-5

In [ ]:
tracer = TracIn(mlp, X_train, y_train, n_points=5, n_ticks=2000, rng=rng, reset_optimizer=True)

In [ ]:
grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

In [ ]:
loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

In [ ]:
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

In [ ]:
plot_centers = True
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

In [ ]:
plot_centers = True
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 4), ncols=4)
tracer.plot(tracin_0, ax=axs[0])
tracer.plot(tracin_1, ax=axs[1])
tracer.plot(tracin_0 + tracin_1, ax=axs[2])
tracer.plot(tracin_0 + tracin_1, ax=axs[3], overlay=True)

# 900 grid points

In [ ]:
mlp._opt_lr = 1e-5
tracer = TracIn(mlp, X_train, y_train, n_points=30, n_ticks=2000, rng=rng, reset_optimizer=True)

In [ ]:
grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

In [ ]:
loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

In [ ]:
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

In [ ]:
plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
plot_centers = False
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)